<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

#  Working With Time Series Data

---

### About
Explore time series data in pandas and measurements and statistics that are relevant to time series data.

### Learning Objectives
- Identify time series data.
- Use the `datetime` library to represent dates as objects.
- Preprocess time series data in pandas.
- Identify stationary time series and difference the data in order to coerce stationarity.
- Understand autocorrelation and partial autocorrelation and intuit why it might exist in a time series.

### Notebook Guide
- Imports and Loading Data
- Dates and Times in pandas
    - The DateTime library
    - Timestamps
    - Filtering Time Series
    - Aggregating Time Data: Resampling
    - Aggregating Time Data: Rolling Statistics
- Backshifting and Diffing
- Stationarity
- Autocorrelation: ACF and PACF
- Bonus Challenge
- Conclusions and Takeaways

# Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

## Stock Data
We'll use [Yahoo! Finance](https://finance.yahoo.com/quote/AAPL) to get a few years worth of stock prices from Apple, Inc. (AAPL)

In [ ]:
# Load data.


# Dates and Times in pandas
---

## The DateTime library

Python's `DateTime` library is great for dealing with time-related data, and pandas has incorporated this library into its own `datetime` series and objects.

## Date Preprocessing 
Typically, when we read in our data from a file it will come in as a python object or string. We will first convert our date columns to the special datetime type using [pd.datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html). This will will give us a lot of date functionality that isn't possible with a regular string object. 

The typical format is as follows: 

```python 
   df["date_column"] =  pd.to_datetime(df["date_column"])
```


In [ ]:
# Overwrite the original `Date` column with one that's been converted to a `datetime` series.



## The `.dt` Attribute
pandas `datetime` columns have a `.dt` attribute that allows you to access attributes that are specific to dates. For example:

```python
    aapl["Date"].dt.day
    aapl["Date"].dt.month
    aapl["Date"].dt.year
    aapl["Date"].dt.hour
    aapl["Date"].dt.minutes
    aapl["Date"].dt.second
    
    appl["Date"].dt.quarter
```


In [ ]:
# Find the Year for our data 

# Optional depending on your needs you can save this as new column


In [ ]:
# Check your work


In [ ]:
# Find months for each 


## Now You Try

In [ ]:
# Find the quarter for each row and save it to a new column "Quarter"
aapl["Quarter"]= aapl["Date"].dt.quarter

In [ ]:
# Check your work
aapl[["Date", "Quarter"]].head(10)

## Fiscal Years and Quarters
It is common to have fiscal years that start in months other than January. The code below takes advantage of date offsets that are built into pandas to help with these common cases. They can also be used to change start of weeks, calculate only business days, and etc. 

```python
   apple_stock["Date"].dt.to_period('Q-JUN')
```

The month listed is the **last month of the fiscal year**. 
- 'Q-MAR' indicates April is the start of the fiscal year
- 'Q-JUN' indicates July 
- ... and so forth 

this will give you an object that has quarter and fiscal year 
```
   0      2017Q3
   1      2017Q3
   2      2017Q3
   3      2017Q3
   4      2017Q3
```
from there you can call the fiscal year or the quarter using your regular: 
```python
    #for quarter
    apple_stock["Date"].dt.to_period('Q-JUN').dt.quarter 
    #for fiscal year
    apple_stock["Date"].dt.to_period('Q-JUN').dt.qyear
```

#### Docs:

- [Docs for common offsets](https://pandas.pydata.org/docs/user_guide/timeseries.html#dateoffset-objects)
- [Docs with list of offsets](https://pandas.pydata.org/docs/user_guide/timeseries.html#anchored-offsets)
- [Docs for to_period](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.to_period.html)

In [ ]:
# Check our work


In [ ]:
# Change date column to be datetime dtype


## Now You Try
Using a new copy of our data `fiscal_df`. Create a column `Year_Quarter` that has the fiscal year and quarter in the following format. Your fiscal year starts in Oct. 

In [ ]:
# Check your work
# fiscal_df[["Date", "Year_Quarter"]].sample(10, random_state=1).head()

## Timestamps

Timestamps are useful objects for comparisons. You can create a timestamp object using the `pd.to_datetime()` function and a string specifying the date. These objects are especially helpful when you need to perform logical filtering with dates.

The main difference between a `datetime` object and a timestamp is that timestamps can be used as comparisons.

## Now You Try

In [ ]:
# Create a date ts1 and filter the apple stock data 



In [ ]:
# filter the data by ts1


## Set the `Date` column to be the index

Many time series methods and functions require the pandas object's index to be the datetime. Thankfully this is a quick one-liner.

In [ ]:
# We generally want our index to be sorted.



Note: you can also revert back to the original index with [.reset_index()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html)

## Filtering Time Series

Now that our `Date` column is the index, we can filter our data in unique ways. Try `aapl.loc['2018']` in the cell below:

You can also filter by month:

In [ ]:
# Pull out all January 2019 data.


In [ ]:
# Pull out all January data, across all years.
# Note this was always possible with datetime objects


## Aggregating Time Data: Resampling

`df.resample()` is similar to `df.groupby()`, but with dates instead of categories.

In [ ]:
# For each month 'ME', calculate the average of
# each numeric value.


In [ ]:
# Calculate the sum over each day.


# What do you notice?

In [ ]:
# Calculate the sum over each quarter.


In [ ]:
# Calculate the sum over each quarter.
# BUT: change the index to be the start of each quarter.


## Aggregating Time Data: Rolling Statistics

With time series, we can "roll" statistics across time. For example, the rolling mean is the mean of a moving window across time periods.

We'll use a `.rolling()` method with a statistical function chained to it – just like with `.resample()` earlier. 

#### The Rolling Mean
For example, the 3-day rolling mean is calculated as:

$$ M_t = \frac{1}{3}\left(X_t + X_{t-1} + X_{t-2}\right) $$

In [ ]:
# Calculate the rolling average of the last 3 days.
# Notice the NAs generated!


## Now You Try

Create a line chart for our data that includes:

* The adjusted closing price for AAPL
* The 12-day moving average of the closing price (the "fast" line)
* The 26-day moving average of the closing price (the "slow" line)

**Fun Fact:** You've just created a MACD chart - a simple trading strategy is to buy or sell a stock when the fast and slow line cross!

---

To the slides!

---

# Backshifting and Diffing
---
**Backshifting** values in a column is simply sliding all the values back a day. It's the operation, for each day $t$, of doing:

$$BX_t = X_{t - 1}$$
 
For example:

| Date | Price | Backshifted Price |
| --- | --- | --- |
| Jan 1 | 100 | _NaN_ |
| Jan 2 | 110 | 100 |
| Jan 3 | 105 | 110 |
| Jan 4 | 108 | 105 |

**Vocabulary:** Backshifting is sometimes called "lagging" and the new columns called "lags". For example, the new column created in the above example is the "first lag".

In [ ]:
# Create a new column that's the backshifted "Open" column


**Differencing** a column is taking today's value and subtracting yesterday's. Another way of putting it is subtracing today's value from today's _backshifted_ value. Mathematically, this is:

$$
\begin{align}
\nabla X_t &= X_t - X_{t - 1} \\
           &= X_t - BX_t
\end{align}
$$

For example:

| Date | Price | Backshifted Price | Diffed Price |
| --- | --- | --- | --- |
| Jan 1 | 100 | _NaN_ | _NaN_ |
| Jan 2 | 110 | 100 | 10 |
| Jan 3 | 105 | 110 | -5 |
| Jan 4 | 108 | 105 | 3 |

In [ ]:
# Create a new column that's the diffed "Open" column.
# Then, create a new column that's "Open" minus the lagged column we made above. What do you notice?


## Now You Try

Create two **separate** plots:
* The closing price of AAPL over time
* The diffed closing price of AAPL over time

Specifications:
* Make sure the _x_-axes line up
* It must be one image with two separate graphs

---

To the slides!

---

# Stationarity
---

First, some new data sets.

In [ ]:
# Population of Australia 1971-1994
aus = pd.read_csv('../data/residents.csv')
aus.set_index(pd.to_datetime(aus.date), inplace=True)
aus.drop(columns='date', inplace=True)
aus.head()

In [ ]:
# Mean annual temperature (Fahrenheight) in New Haven, CT 1912-1971


Let's plot these time series. Are they stationary? (We're just eye-balling stationarity for now.)

If not... can you achieve stationarity after diffing?

In [ ]:
# Absolutely not! Clear upward trend.


In [ ]:
# Yeah, kinda!


In [ ]:
# Maybe! Seems trendless enough.


In [ ]:
# Definitely after diffing once!


---

To the slides!

---

# Autocorrelation: ACF and PACF
---
Explore the ACF and PACF graphs for both `aus` and `nh`. What do you notice about them?

The Australian population increases quickly over time - so we don't see any seasonal autocorrelation, but we do some steadily decreasing autocorrelation implying the upward trend. We also see only lag-1 partial autocorrelation since the lag-1 autocorrelation is independent of any lag-2 correlation.

The annual temperature in New Haven, CT doesn't increase or decrease over time and doesn't exhibit any seasonality. This is exhibited by no detectable [partial] autocorrelations.

## You Try: Lung disease data

Take a look at the lung disease data imported below. Look at its plot, ACF, and PACF. Can you explain the notable parts of each graph?

## BONUS CHALLENGE

For each year 1974 to 1979, plot the lung disease-related deaths on the same 12-month _x_-axis. Do you notice any interesting patterns?

Yes! It appears that more deaths occur in colder months.

# Conclusions and Takeaways

* A **time series** is a series of observations taken in equally spaced intervals. Often this means days, months, quarters, or years.
* We have **dependent data** - we're very much concerned with row-to-row relationships and operation.